In [1]:
import dfBasics
import common
import encoder
import pfAdapt
import charts

Setup Complete


In [2]:
import pandas as pd
from pyspark.sql import functions
import pyspark.sql.functions as f

In [3]:
sparkSession = dfBasics.getSparkSession()

## encoded values

### init encoders (run only once)

In [14]:
# - run this paragraph only once !
# - if you get an 'allow_pickle' error you need Kernel/restart kernel

import encoder
import numpy as np

np_load_old = np.load

# modify the default parameters of np.load
np.load = lambda *a,**k: np_load_old(*a, allow_pickle=True, **k)

# restore np.load for future normal usage
#np.load = np_load_old

### functions

In [15]:
import numpy

def get_encoders(columns,npy):
    encoders = {}
    for column in columns:
        _encoder = encoder.TolerantLabelEncoder(ignore_unknown=True)
        _encoder.classes_ = np.load(npy + '/' + column + '.npy')
        encoders[column] = _encoder
    return encoders    

def get_list(encoders,column):
    return list(encoders[column].classes_)
    
def transform(value,_encoder ):
    try:
        return int( _encoder.transform([value])[0])
    except Exception as e:
        return -1
    
def inverse_transform(value,_encoder):
    if type(value) in [int,numpy.int64]:
        return str(_encoder.inverse_transform(value))  
    elif type(value) == list:
        return [str(_encoder.inverse_transform(v)) for v in value]    

## Main

### init

In [16]:
columns = ['CSTATUS', 'CSERVICE', 'CSENDERENDPOINTID', 'CSENDERPROTOCOL','CRECEIVERPROTOCOL', 'CRECEIVERENDPOINTID']

share_directory =  '/home/jovyan/work/share/sla/'
share_directory = '/home/jovyan/work/output/'

version_sla_2 = 'v00002'
version_2     = version_sla_2 + '/v00001'
version_sla_4 = 'v00004'
version_4     = version_sla_4 + '/v00000'

encoders_2 = get_encoders(columns,npy= share_directory  + version_sla_2 + '/npy')
senders_2 = get_list(encoders_2,'CSENDERENDPOINTID')
encoders_4 = get_encoders(columns,npy= share_directory  + version_sla_4 + '/npy')
senders_4 = get_list(encoders_4,'CSENDERENDPOINTID')

### test encoding

In [17]:
inverse_transform(6738, encoders_4['CRECEIVERENDPOINTID'])
transform('c5956260-e0ee-11e8-be62-528eac1b495c', encoders_4['CRECEIVERENDPOINTID'])

6738

In [44]:
sender_receivers_df = pd.read_parquet('/home/jovyan/work/output/v00004/single/' + 'sender_receivers.parquet')
sender_receivers_df 

,CRECEIVERENDPOINTID
3914,"[6738, 2189, 3526, 6778, 3610, 420, 8114, 1157..."
2863,"[6778, 3610]"
946,"[6778, 3610, 3354, 2428]"
4926,"[6738, 2189, 3526, 420, 7143, 3544, 4649, 984,..."
6222,[6911]
...,...
3879,[2919]
3309,"[3354, 6778, 5008, 7736, 2428]"
5370,"[4001, 4927, 4058, 6724, 2565, 3824, 5226, 663..."
2142,[5255]


In [40]:

#sender_receivers_df

In [41]:
import glob
sender_receivers_df = pd.DataFrame(columns=['CRECEIVERENDPOINTID'])
base_dir = '/home/jovyan/work/output/v00004/single/sender_receivers/'
filenames = glob.glob(base_dir + '*')
for filename in filenames:
    sender = filename.split('/')[-1].split('.')[0]
    df = sparkSession.read.parquet(filename).toPandas()
    encoded_sender = transform(sender, encoders_4['CSENDERENDPOINTID'])
    encoded_receivers = [transform(receiver,encoders_4['CRECEIVERENDPOINTID']) for receiver in list(df['CRECEIVERENDPOINTID']) if receiver is not None]
    sender_receivers_df.loc[encoded_sender] = [encoded_receivers]

In [43]:
sender_receivers_df.to_parquet('/home/jovyan/work/output/v00004/single/' + 'sender_receivers.parquet')

In [1]:
!pwd

/home/jovyan/work/scray/scray-examples/python
